In [5]:
import praw
import pandas as pd
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk

# nltk.download('punkt')
# nltk.download('stopwords')

# Initialize Reddit API
reddit = praw.Reddit(
    client_id="61G5r3qum5peFaRdvtbMJg",
    client_secret="yPzjWBhh797nj2muWQJt3jAEhtZtqg",
    user_agent="script by u/Educational_Leek_918"
)

# Function to scrape subreddit data
def scrape_reddit(subreddit_name, limit=1000):
    subreddit = reddit.subreddit(subreddit_name)
    posts = []
    for post in subreddit.new(limit=limit):
        posts.append({
            "title": post.title,
            "body": post.selftext,
            "created_utc": post.created_utc,
            "upvotes": post.score,
            "num_comments": post.num_comments
        })
    return pd.DataFrame(posts)

# Scrape data
subreddit_data = scrape_reddit("stocks", limit=1000)

# Save raw data
subreddit_data.to_csv("reddit_stocks_raw.csv", index=False)
print(f"Scraped {len(subreddit_data)} posts!")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\vardh\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\vardh\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Scraped 832 posts!


In [6]:
# Function to clean text data
def clean_text(text):
    text = re.sub(r'http\S+|www.\S+', '', text)  # Remove URLs
    text = re.sub(r'[^a-zA-Z\s]', '', text)      # Remove special characters
    text = text.lower()                          # Convert to lowercase
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word not in stopwords.words('english')]
    return ' '.join(tokens)

# Apply cleaning
subreddit_data["cleaned_title"] = subreddit_data["title"].apply(clean_text)
subreddit_data["cleaned_body"] = subreddit_data["body"].apply(clean_text)

# Save cleaned data
subreddit_data.to_csv("reddit_stocks_cleaned.csv", index=False)
print("Data cleaned and saved!")

Data cleaned and saved!


In [9]:
from textblob import TextBlob

# Function for sentiment polarity
def sentiment_score(text):
    return TextBlob(text).sentiment.polarity

# Add sentiment scores
subreddit_data["title_sentiment"] = subreddit_data["cleaned_title"].apply(sentiment_score)
subreddit_data["body_sentiment"] = subreddit_data["cleaned_body"].apply(sentiment_score)

# Save data with sentiment
subreddit_data.to_csv("reddit_stocks_sentiment.csv", index=False)
print("Sentiment analysis complete and saved!")

Sentiment analysis complete and saved!


In [10]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report
import numpy as np

# Feature and label preparation
subreddit_data["sentiment"] = (subreddit_data["title_sentiment"] + subreddit_data["body_sentiment"]) / 2
subreddit_data["price_movement"] = np.random.choice([0, 1], size=len(subreddit_data))  # Dummy target

X = subreddit_data[["sentiment"]]
y = subreddit_data["price_movement"]

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

# Evaluate model
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.46706586826347307
Precision: 0.4626865671641791
Recall: 0.36904761904761907

Classification Report:
               precision    recall  f1-score   support

           0       0.47      0.57      0.51        83
           1       0.46      0.37      0.41        84

    accuracy                           0.47       167
   macro avg       0.47      0.47      0.46       167
weighted avg       0.47      0.47      0.46       167

